In [0]:
from pyspark.sql.functions import *
import time
import requests, json, re

In [0]:
CANDIDATE_PROFILE = """""Professional Summary
Detail-oriented Entry-Level Verification Engineer skilled in Digital Electronics, Verilog, SystemVerilog, UVM, and Tcl scripting. Hands-on experience developing testbenches, writing Assertions (SVA), and defining functional coverage across complex interfaces, including DDR4, AHB, APB, I2C, and AHB-to-APB bridges. 
Technical Skills
•	Programming: Verilog, SystemVerilog, UVM, Basics of UVM RAL.
•	Scripting: Basics of Tcl
•	Protocol: APB, AHB, I2C, DDR4, Basics of AXI.
•	Simulation Tools: Siemens Questa Sim, Synopsys (VCS), EDA Playground.
•	Version Control: Basics of GIT.
Projects
DDR4 Verification
•	Developed a UVM-based verification environment using Siemens QuestaSim for a high-performance DDR4 Memory Controller, verifying MPR, FGR, ODT timing, JEDEC constraints, burst modes, and back-to-back read/write traffic using Functional Coverage and SVA. Validated data integrity, timing compliance, reset behavior, and protocol error scenarios.
APB Design and Verification
•	Designed and verified a synthesizable APB Master/Slave Interface in Verilog using UVM and Siemens QuestaSim, covering Setup/Access phases, PSEL/PENABLE handshakes, PREADY wait states, and PSLVERR errors through Functional Coverage and SVA. Validated continuous read/write transfers, reset behavior, data integrity, and error scenarios.
I2C Design and Verification
•	Designed a synthesizable I2C Single-Master Single-Slave Controller in Verilog and developed a UVM-based verification environment using Siemens QuestaSim, verifying Start/Stop conditions, ACK/NACK handshakes, data transfers, and 100 kHz operation using Functional Coverage and SVA. Validated reset, error handling, boundary conditions, and different transfer scenarios.
AHB to APB Bridge Verification
•	Developed a UVM environment using Synopsys VCS and Verdi to verify an AMBA AHB-to-APB Bridge, covering protocol conversion, handshakes, transfer types, wait states, boundary conditions, and error scenarios using Functional Coverage and SVA. Validated AHB-to-APB transactions, reset behavior, and data integrity across different traffic patterns.
"""

In [0]:
p1 = """
You are an expert technical recruiter and ATS evaluator.
  Task:
  Compare the provided Job Description (JD) against the Candidate Profile and calculate a match score between 0 and 100. Also identify the company type (product based or Not product based) based on the company name.
  Job Description:
"""

p2 = "Candidate Profile:" + CANDIDATE_PROFILE

format = """
CRITICAL OUTPUT REQUIREMENTS:
          -Return ONLY valid JSON.
            {
            "score": <integer between 0 and 100>
            "company_type": "P/NP"
            }
          - Do NOT return the steps you followed, just return the json
          - Do NOT include markdown.
          - Do NOT wrap JSON in ```json blocks.
          - Do NOT provide explanations outside the JSON.
          - Do NOT include introductory text such as "Here is the result" or "To determine the match score".
          - The response must start with { and end with }.
          - If any field is unknown, use null.
          - Output must be parseable by json.loads() without modification.
          - Ex1: {"score":88, "company_type": "NP"}
          - Ex2: {"score":67, "company_type": "P"}
"""
rules = """
Mandatory Eligibility Rule:
          1. From JD Extract the minimum years of total experience required as min_exp_required.
          2. If the JD explicitly requires more than 1 years of experience (min_exp_required > 1), return: 0
          3. Do not continue skill matching if Rule #3 is triggered.
          
          Scoring Rules (only if eligible):
          1. Compare candidate skills, experience, tools, technologies, and domain knowledge against the JD.
          2. Assign scores:
          - Required skills match: 70%
          - Preferred/Nice-to-have skills match: 20%
          - Relevant domain/project experience: 10%
          3. Calculate a final score between 0 and 100.
          4. Be strict. Do not award points for unrelated skills.
"""

In [0]:
def send_message(message, chat_ids):
    TOKEN = dbutils.secrets.get("job_notification", "Telegram_Token")
    CHAT_ID = chat_ids
    MESSAGE = message

    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
    for user in CHAT_ID:
        payload = {
            "chat_id": user,
            "text": MESSAGE,
            "parse_mode": "HTML"
        }

        res = requests.post(url, data=payload)

In [0]:
# def form_message_and_send(combined_df):
#     df = spark.table('main_catalogue.jobs.users')
#     chat_ids = df.filter(df.skill == "Data Engineer").select(collect_list('chat_id')).collect()[0][0]
#     s = ''
#     count = 1
#     tc = 1
#     tl = combined_df.count()
#     print(tl)
#     for row in combined_df.collect():
#         # print(row)
#         flag = ""
#         if row["company_type"] == "P":
#             flag = "(P)"
#         s += f"""{tc}. <a href = "{row["url"]}" >{row["title"]}</a> {flag}\n<b>{row["company"]}</b>, <i>{row["location"]}</i> \n\n"""
#         if count==20 or tc==tl:
#             print(s)
#             send_message(s, chat_ids)
#             count = 0
#             s = ''
#         count+=1
#         tc+=1

In [0]:
def form_message_and_send(table):
    df = spark.table('raw_catalogue.vlsi_jobs.users')
    jobs_df = spark.table(table)
    chat_ids = df.select(collect_list('chat_id')).collect()[0][0]
    s = ''
    count = 1
    tc = 1
    jobs = jobs_df.collect()
    tl = len(jobs)
    print(tl)
    l = []
    for row in jobs:
        # print(row)
        flag = ""
        title = "Title"
        if row["company_type"] == "P":
            flag = "(P)"
        if row['title'].strip():
            title = row['title']
        s += f"""{tc}. <a href = "{row["url"]}" >{title}</a> {flag}\n<b>{row["company"]}</b>, <i>{row["location"]}</i> \n\n"""
        if count==20 or tc==tl:
            # print(s)
            send_message(s, chat_ids)
            count = 0
            s = ''
        count+=1
        tc+=1